# Notebook 04 - SFT With Adaption-Improved Data

This is the adapted-data SFT condition. It trains the same base model with the same training settings as Notebook 03, but uses `data/processed/kakugo_adapted_sft.jsonl`.

Keeping these notebooks parallel makes the comparison easier to reason about.

**Files read:**
- [`../configs/tinker_sft_adapted.yaml`](../configs/tinker_sft_adapted.yaml) - base model, renderer, training hyperparameters, input path, and output path for the adapted-data SFT run.
- [`../data/processed/kakugo_adapted_sft.jsonl`](../data/processed/kakugo_adapted_sft.jsonl) - chat-format adapted SFT examples prepared in Notebook 02.

**Files written:**
- [`../results/models/sft_adapted/`](../results/models/sft_adapted/) - Tinker logs, metrics, checkpoints, and final model/sampler references for the adapted-data SFT run.

In [ ]:
import asyncio
import json
import logging
import os
import sys
import warnings
from pathlib import Path

import nest_asyncio
from huggingface_hub import login
from tinker_cookbook.supervised.data import FromConversationFileBuilder
from tinker_cookbook.supervised.train import Config, main
from tinker_cookbook.supervised.types import ChatDatasetBuilderCommonConfig

nest_asyncio.apply()
warnings.filterwarnings('ignore', module='tinker_cookbook')

In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError(
        'Run this notebook from inside the adaption-kirundi-sft-starter repo.'
    )


ROOT = find_repo_root(Path.cwd())
SRC = str(ROOT / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

In [ ]:
from kirundi_sft_starter.utils import ensure_dir, load_env, load_yaml, require_file

load_env()
adapted_sft_config = load_yaml(ROOT / 'configs/tinker_sft_adapted.yaml')

require_file(ROOT / adapted_sft_config['data_path'], 'Run Notebook 02 before launching adapted-data SFT.')
ensure_dir(ROOT / adapted_sft_config['output_dir'])

if not os.environ.get('TINKER_API_KEY'):
    raise RuntimeError('Missing TINKER_TOKEN or TINKER_API_KEY. Add it to .env before launching training.')

if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'])

## Review the training config

This is the exact adapted-data SFT configuration that will be sent to Tinker in the training cell below.

In [ ]:
print(json.dumps(adapted_sft_config, indent=2))

## Build the Tinker SFT config

This mirrors Notebook 03. The only intentional difference is the data path and output directory from `configs/tinker_sft_adapted.yaml`.

In [ ]:
renderer_name = adapted_sft_config['renderer_name']

common_config = ChatDatasetBuilderCommonConfig(
    model_name_for_tokenizer=adapted_sft_config['base_model'],
    renderer_name=renderer_name,
    max_length=int(adapted_sft_config['max_length']),
    batch_size=int(adapted_sft_config['batch_size']),
    train_on_what=adapted_sft_config['train_on_what'],
)

dataset_builder = FromConversationFileBuilder(
    file_path=str(ROOT / adapted_sft_config['data_path']),
    common_config=common_config,
)

sft_config = Config(
    log_path=str(ROOT / adapted_sft_config['output_dir']),
    model_name=adapted_sft_config['base_model'],
    dataset_builder=dataset_builder,
    learning_rate=float(adapted_sft_config['learning_rate']),
    lora_rank=int(adapted_sft_config['lora_rank']),
    num_epochs=int(adapted_sft_config['num_epochs']),
)

print('\nSFT Config:')
print(f"  Run:           {adapted_sft_config['run_name']}")
print(f"  Model:         {sft_config.model_name}")
print(f"  Renderer:      {renderer_name}")
print(f"  Data:          {ROOT / adapted_sft_config['data_path']}")
print(f"  Output:        {sft_config.log_path}")
print(f"  Learning rate: {sft_config.learning_rate}")
print(f"  LoRA rank:     {sft_config.lora_rank}")
print(f"  Epochs:        {sft_config.num_epochs}")

## Launch training

This cell starts the real Tinker SFT job. If credentials, package imports, data paths, or Tinker access are wrong, the cell should fail loudly so the issue is visible.

In [ ]:
print('=' * 50)
print('Starting adapted-data SFT training...')
print('Watch train_mean_nll; it should generally decrease across training.\n')

logging.getLogger('asyncio').setLevel(logging.CRITICAL)
result = asyncio.get_event_loop().run_until_complete(main(sft_config))

print('\nTraining complete!')
print(f'Result: {result}')